In [1]:
import pandas as pd
s = pd.read_csv("data/sources.csv", keep_default_na=False)
print(s[s["dataset"].str.startswith("Safety")][["dataset", "file"]].to_string())

                                                  dataset                                       file
5          Safety document: American-Red-Cross-Safety.txt         docs\American-Red-Cross-Safety.txt
6          Safety document: FEMA-Earthquake-Guide.txt.txt         docs\FEMA-Earthquake-Guide.txt.txt
7           Safety document: KMA-Korea-Disaster-Guide.txt          docs\KMA-Korea-Disaster-Guide.txt
8          Safety document: Ready-Gov-Earthquake-Prep.txt         docs\Ready-Gov-Earthquake-Prep.txt
9             Safety document: USGS-Earthquake-Basics.txt            docs\USGS-Earthquake-Basics.txt
10  Safety document: earthquake-safety-knowledge-base.txt  docs\earthquake-safety-knowledge-base.txt


In [ ]:
import os, pandas as pd

s = pd.read_csv("data/sources.csv", keep_default_na=False)
fill = {
  "American-Red-Cross": (
      "American Red Cross",
      "https://www.redcross.org/get-help/how-to-prepare-for-emergencies/types-of-emergencies/earthquake.html",
      "Copyrighted by the American Red Cross (https://www.redcross.org/terms-of-use). Private study use only; no permission requested for reproduction; web page shows only short snippets with a link."
  ),

  "FEMA-Earthquake": (
      "FEMA",
      "https://www.fema.gov/sites/default/files/2020-08/fema_earthquakes_fema-p-530-earthquake-safety-at-home-march-2020.pdf",
      "FEMA.gov: most material free of copyright, may be copied without permission, citation and link requested (https://fema.gov/node/367589). Check this document for third-party material."
  ),

  "KMA-Korea-Disaster": (
      "Korea Meteorological Administration",
      "https://www.kma.go.kr/kma/biz/earthquake_volcano_03.jsp",
      "KMA copyright policy: free use only for works marked KOGL Type 1, source must be cited (https://www.kma.go.kr/kmadev/guide/copyright.jsp). KOGL mark checked: no."
  ),

  "Ready-Gov": (
      "Ready.gov (FEMA)",
      "https://www.ready.gov/earthquakes",
      "Ready Campaign publications free to download; content must not be altered. Text kept verbatim."
  ),

  "USGS-Earthquake": (
      "U.S. Geological Survey",
      "https://www.usgs.gov/programs/earthquake-hazards/prepare",
      "USGS-authored content is U.S. public domain, credit requested (https://www.usgs.gov/information-policies-and-instructions/copyrights-and-credits). Some images are third-party."
  ),

  "earthquake-safety-knowl": (
      "Own compilation from the rows above",
      "n/a",
      "Own compilation from the sources above."
  ),
}

for key, (pub, url, terms) in fill.items():
    m = s["file"].str.contains(key, case=False, regex=False)
    if not m.any(): print("no row for", key); continue
    s.loc[m, ["publisher", "url", "license_terms"]] = [pub, url, terms]

# GDP row: create the file from value_per_person.csv if it is missing
if not os.path.exists("data/gdp_per_capita.csv"):
    v = pd.read_csv("data/value_per_person.csv", keep_default_na=False, na_values=[""])
    keep = [c for c in ["iso3","iso2","name","region","income","year","gdp_pc","gdp_estimated","estimated"] if c in v.columns]
    v[keep].to_csv("data/gdp_per_capita.csv", index=False)
if not (s["file"] == "data/gdp_per_capita.csv").any():
    s.loc[len(s)] = ["GDP per capita", "data/gdp_per_capita.csv", "World Bank",
        "https://data.worldbank.org/indicator/NY.GDP.PCAP.CD", "PASTE the real download date",
        "CC BY 4.0, credit World Bank", "Value per person",
        "Indicator NY.GDP.PCAP.CD (current US dollars), latest year per country"]

left = s[s.apply(lambda r: any(x in " ".join(map(str, r.values)) for x in ("PASTE", "TODO", "TO CHECK")), axis=1)]
if len(left): print("Still to fill:", left["dataset"].tolist())
else: s.to_csv("data/sources.csv", index=False); print("saved")

catalog.csv already exists, not rebuilding
  (no row added, file not found: data/usgs_raw.csv)
sources.csv: 12 rows (0 added)

PASS  catalog.csv - 124,304 rows, 2000-01-01 to 2026-11-20, sources {'EMSC': 124304}
PASS  catalog covers the world - latitude -78 to 87, longitude -180 to 180
PASS  city table - 160,223 places, South Korea total 47,941,254
PASS  value per person - 217 countries, median 161,695 USD per person
PASS  damage-ratio table - 212 countries, Japan loss ratio at MMI IX = 0.090 (report: almost 10 percent)
FAIL  sources.csv complete - 6 rows still need work: Safety document: American-Red-Cross-Safe, Safety document: FEMA-Earthquake-Guide.t, Safety document: KMA-Korea-Disaster-Guid, Safety document: Ready-Gov-Earthquake-Pr, Safety document: USGS-Earthquake-Basics., Safety document: earthquake-safety-knowl

OVERALL: NOT DONE YET - fix the FAIL lines above
